# S10 — does physics help BEYOND an improved (EMA, 2-seed) baseline?
Rules: `S10_preregistration.txt` (fixed before any S10 code ran). CMMD, trained from scratch, 50 epochs, EMA, seeds 42 + 43:
**E0** = baseline (λ 0) · **E1** = baseline + noise-physics loss on x̂₀ (λ 0.01).
**K1** EMA vs raw · **K2 (primary)** E1-EMA vs E0-EMA · **K3** E1-raw vs E0-raw · D2 epoch 25 vs 50 (descriptive).

**Session 1 input:** the **S8 replication** notebook output (latest version; provides `cmmd_data/` and `cmmd_clf_s0.pt`).
**Session 2 (and 3):** the same input **plus this notebook's own latest output**. GPU T4 ×2 · Internet ON ·
Save & Run All (Commit). ≈ 11 h + 11 h + ~2 h. Send `S10_results.zip` (or the log) after each session.

In [ ]:
# ===================== CELL 1 — CONFIG =====================
BUDGET_H = 11.0
EPOCHS = 50
VAL_N, TEST_N = 100, 150
VAL_SEED, TEST_SEED = 2025, 9025
TEST_ALPHA = 0.05 / 3              # 98.33 % CI = Bonferroni over K1-K3
RUN_SMOKE = True
import os, time
os.environ["PA_OUT"] = "/kaggle/working/S10"
os.makedirs(os.environ["PA_OUT"], exist_ok=True)
if not os.path.exists("/tmp/pg_session_start"):
    open("/tmp/pg_session_start", "w").write(str(time.time()))
def used_h():
    return (time.time() - float(open("/tmp/pg_session_start").read())) / 3600
print("S10 | budget", BUDGET_H, "h | output", os.environ["PA_OUT"])

In [ ]:
# ===================== CELL 2 — setup (needs Internet ON) =====================
import os, re, sys, subprocess
os.chdir("/kaggle/working")
if not os.path.isdir("/kaggle/working/U24_synthesis/.git"):
    subprocess.run("git clone -q https://github.com/Its-Me-Sk/U24_synthesis.git", shell=True, check=True)
if not os.path.exists("/tmp/pip_done"):
    # Kaggle runs Python 3.13: install the repo requirements once, never force-reinstall numpy / pandas
    subprocess.run("cd /kaggle/working/U24_synthesis && grep -v xformers requirements.txt > /tmp/req.txt && "
                   "pip install -q -r /tmp/req.txt", shell=True, check=True)
    subprocess.run("pip install -q --no-deps torch-fidelity", shell=True, check=True)
    open("/tmp/pip_done", "w").write("ok")
probe = ("import sys, numpy, pandas, scipy, torch, torchmetrics, monai, generative, PIL; "
         "print('python', sys.version.split()[0], '| numpy', numpy.__version__, '| pandas', pandas.__version__, "
         "'| torch', torch.__version__, '| torchmetrics', torchmetrics.__version__, '| monai', monai.__version__)")
env_chk = subprocess.run([sys.executable, "-c", probe], capture_output=True, text=True)
if env_chk.returncode != 0:
    print(env_chk.stderr[-2500:])
    raise SystemExit("!! environment check failed (see the error above) -> send Claude this block")
print("environment OK:", env_chk.stdout.strip())
p = "/kaggle/working/U24_synthesis/lightning_synthesis/model_architectures/monai_ddpm.py"
s = open(p).read()
s2 = re.sub(r"use_flash_attention\s*=\s*True", "use_flash_attention=False", s)   # xformers is not installed
if s2 != s:
    open(p, "w").write(s2)
print("flash attention ->", "patched off" if s2 != s else "already off")
m = "/kaggle/working/U24_synthesis/models.py"          # oracle loader (used later in A6): hard-coded config path
s = open(m).read()
s2 = re.sub(r"open\(\s*['\"]/home/[^'\"]*config\.yaml['\"]", "open('/kaggle/working/U24_synthesis/config.yaml'", s)
if s2 != s:
    open(m, "w").write(s2)
print("Cell 2 done. Kernel restart NOT needed (all GPU work runs in subprocesses).")

In [ ]:
# ===================== CELL 3 — code + CMMD cache (reused from S8) + carry-forward + runner =====================
import os, sys, glob, json, shutil, subprocess, threading, time
from pathlib import Path
WD = "/kaggle/working/U24_synthesis/lightning_synthesis"
os.chdir(WD)
FILES = {"phaseA_common.py": r'''# phaseA_common.py - shared helpers for the Phase A diagnostics (no training happens in Phase A).
# Model, schedule, data domain and conditioning are taken from the Experiment-2 code (train_arm2.py / eval_arm.py):
#   * model: model_architectures.MonaiDDPM().unet (MONAI-Generative DiffusionModelUNet, flash attention off)
#   * schedule: MonaiDDPM().scheduler.alphas_cumprod (linear beta 1e-4..0.02, T=1000)
#   * images: per-image min-max to [-1,1]; physics statistic on (x+1)/2
#   * conditioning: context = label as float (B,1,1); -1 = null label (classifier-free guidance)
import os, sys, glob, json, math, time
import numpy as np
import torch

WD = "/kaggle/working/U24_synthesis/lightning_synthesis"
if WD not in sys.path:
    sys.path.insert(0, WD)
OUT = os.environ.get("PA_OUT", "/kaggle/working/phaseA")
os.makedirs(OUT, exist_ok=True)


def log(tag, *m):
    print(f"[{tag}]", *m, flush=True)


_INDEX = None


def find(name, roots=("/kaggle/input", "/kaggle/working")):
    """all files with this exact name in the attached inputs (one directory walk per process; image folders skipped)"""
    global _INDEX
    if _INDEX is None:
        _INDEX = {}
        for r in roots:
            for dp, dns, fns in os.walk(r):
                dns[:] = [d for d in dns if d not in ("jpeg", "U24_synthesis", ".git", "phaseA", "CMMD")]
                for fn in fns:
                    _INDEX.setdefault(fn, []).append(os.path.join(dp, fn))
    return sorted(set(_INDEX.get(name, [])))


def main_ckpt():
    """full-data Experiment-2 baseline, epoch 40 (the main Phase A checkpoint)"""
    h = find("base2_w_ep040.pt")
    if not h:
        raise SystemExit("base2_w_ep040.pt not found: Add Input -> Your Work -> the Experiment 2 notebook "
                         "(notebook64a7f79592), then run again")
    return h[0]


def clean_ckpt(seed, epoch=40):
    """clean-data baseline (step 2 of the timeline): base_clean_s<seed>_w_ep<epoch>.pt"""
    h = find(f"base_clean_s{seed}_w_ep{epoch:03d}.pt")
    if not h:
        raise SystemExit(f"base_clean_s{seed}_w_ep{epoch:03d}.pt not found: Add Input -> Your Work -> the clean-baseline "
                         "notebook (its latest output version), then run again")
    return h[0]


def optional_25_ckpt():
    """any 25 %-data baseline at epoch 45 from Experiment 3 or 4 (optional)"""
    for s in (42, 43, 44, 45, 46, 47, 48):
        h = find(f"base3_s{s}_w_ep045.pt")
        if h:
            return h[0]
    return None


def schedule(dev):
    from model_architectures import MonaiDDPM
    m = MonaiDDPM(lr=1e-5, T=1000)
    ac = m.scheduler.alphas_cumprod.detach().double().to(dev)
    unet = m.unet
    del m
    return unet, ac


def load_model(path, dev, dtype=torch.float32):
    """returns (unet in eval mode, alphas_cumprod float64, snapshot meta)"""
    snap = torch.load(path, map_location="cpu", weights_only=False)
    unet, ac = schedule(dev)
    sd = snap["unet"] if "unet" in snap else snap["models"]["net"]
    unet.load_state_dict({k: v.float() for k, v in sd.items()})
    unet = unet.to(dev).to(dtype).eval()
    for p in unet.parameters():
        p.requires_grad_(False)
    meta = {k: snap[k] for k in ("epoch", "step", "lam", "tau") if k in snap}
    meta["nlf"] = snap.get("nlf")
    meta["path"] = path
    del snap
    return unet, ac, meta


def load_cache(split):
    d = np.load(os.path.join(os.environ.get("PA_TMP", "/tmp"), f"{split}_cache.npz"))      # S8: dataset switch
    return d["x"], d["y"]


def load_clean_meta(split):
    """clean cache WITH patient ids and md5 - same order as /tmp/<split>_cache.npz (written by prepare_clean.py)"""
    d = np.load(os.path.join(os.environ.get("PA_DATA", "/kaggle/working/clean_data"), f"{split}_cache.npz"))   # S8: dataset switch
    return d["x"], d["y"], d["patient"].astype(str), d["md5"].astype(str)


def val_mask(patients, frac=0.15, seed=0):
    """patient-level VALIDATION split of the clean TRAIN set (classifiers, gates, guidance choice only).
    The diffusion model itself was trained on all of train. Fixed seed -> identical in every script/session."""
    ups = sorted(set(patients))
    rng = np.random.default_rng(seed)
    val = set(rng.choice(ups, int(round(frac * len(ups))), replace=False).tolist())
    return np.array([p in val for p in patients])


def minmax_pm1(x):
    """x: torch (B,1,H,W) any range -> per-image min-max to [-1,1] (training domain)"""
    mn = x.amin(dim=(1, 2, 3), keepdim=True)
    mx = x.amax(dim=(1, 2, 3), keepdim=True)
    return 2 * (x - mn) / (mx - mn + 1e-8) - 1


def to_model_domain(x_u8, dev):
    return minmax_pm1(torch.from_numpy(x_u8).float().unsqueeze(1).to(dev))


def ctx_of(labels, dev, dtype=torch.float32):
    """labels: list/array of 0, 1 or -1 (null) -> context tensor (B,1,1)"""
    return torch.as_tensor(np.asarray(labels, dtype=np.float32), device=dev).view(-1, 1, 1).to(dtype)


def eps_pred(unet, x, t_int, ctx, amp=False, chunk=8):
    """epsilon prediction. amp=True: exactly as in training (fp32 weights + fp16 autocast); amp=False: the weights'
    own dtype. Large batches are split into chunks (the non-flash attention at 64x64 needs ~0.5 GB per image)."""
    if chunk and x.shape[0] > chunk:
        if ctx.shape[0] == 1:
            ctx = ctx.expand(x.shape[0], *ctx.shape[1:])
        return torch.cat([eps_pred(unet, x[i:i + chunk], t_int, ctx[i:i + chunk], amp, None)
                          for i in range(0, x.shape[0], chunk)])
    tb = torch.full((x.shape[0],), int(t_int), device=x.device, dtype=torch.long)
    if amp:
        with torch.autocast("cuda", dtype=torch.float16):
            out = unet(x, timesteps=tb, context=ctx)
        return out.float()
    return unet(x, timesteps=tb, context=ctx)


def save_json(obj, name):
    with open(os.path.join(OUT, name), "w") as fh:
        json.dump(obj, fh, indent=1, default=float)
''',
         "pinn_physics.py": r'''# pinn_physics.py - physics-informed soft constraint (PINN-style residual) for mammogram DDPM.
# Physical model: signal-dependent noise-level function (NLF) of the X-ray film/digitiser chain
#   sigma^2(mu) = b + a*mu + c*mu^2
#   b: Gaussian read/digitiser noise, a*mu: Poisson (quantum) noise, c*mu^2: multiplicative film-grain/gain term
# The coefficients are CALIBRATED on real training images, then the residual between the NLF of a
# generated image and the physical NLF is added to the diffusion loss (lambda * residual).
import torch
import torch.nn.functional as F


class PhysicsNLF:
    def __init__(self, device, k=7, K=16, lo=0.08, hi=0.95, mask_lo=0.04):
        ax = torch.arange(7, device=device, dtype=torch.float32) - 3
        g = torch.exp(-ax ** 2 / 2.0)
        self.g = g / g.sum()
        self.k, self.K, self.lo, self.hi, self.mask_lo = k, K, lo, hi, mask_lo
        self.c = torch.linspace(lo, hi, K, device=device)
        self.bw = float(hi - lo) / (K - 1) * 0.6
        self.p = None                     # (b, a, c) of the calibrated NLF

    def blur(self, I):
        g = self.g
        I = F.conv2d(F.pad(I, (3, 3, 0, 0), mode="reflect"), g.view(1, 1, 1, 7))
        return F.conv2d(F.pad(I, (0, 0, 3, 3), mode="reflect"), g.view(1, 1, 7, 1))

    def stats(self, I):
        # I: (B,1,H,W) intensities in [0,1]. Returns per-image binned high-frequency variance vs local mean.
        h = I - self.blur(I)
        mu = F.avg_pool2d(I, self.k, 1, self.k // 2, count_include_pad=False).flatten(1)
        v = F.avg_pool2d(h * h, self.k, 1, self.k // 2, count_include_pad=False).flatten(1)
        W = torch.exp(-(mu.unsqueeze(-1) - self.c) ** 2 / (2 * self.bw ** 2))
        W = W * (mu > self.mask_lo).unsqueeze(-1).float()        # ignore black background (air)
        mass = W.sum(1)                                           # (B,K)
        vbin = torch.einsum("bpk,bp->bk", W, v) / (mass + 1e-6)   # (B,K)
        return vbin, mass

    def model_var(self):
        b, a, c = self.p
        return (b + a * self.c + c * self.c ** 2).clamp_min(1e-7)

    def residual(self, I):
        # per-image physics residual: mass-weighted squared log-ratio of measured vs physical NLF
        vbin, mass = self.stats(I)
        r = (torch.log(vbin + 1e-5) - torch.log(self.model_var() + 1e-5)) ** 2
        w = mass / (mass.sum(1, keepdim=True) + 1e-6)
        return (w * r).sum(1)

    @torch.no_grad()
    def fit(self, X_u8, bs=32):
        dev = self.c.device
        tv = torch.zeros(self.K, device=dev, dtype=torch.float64)
        tm = torch.zeros(self.K, device=dev, dtype=torch.float64)
        for i in range(0, X_u8.shape[0], bs):
            I = X_u8[i:i + bs].to(dev).float().div(255).unsqueeze(1)
            vb, m = self.stats(I)
            tv += (vb * m).sum(0).double()
            tm += m.sum(0).double()
        V = tv / tm.clamp_min(1e-6)
        wt = tm / tm.sum()
        c = self.c.double()

        def wls(cols):
            A = torch.stack(cols, 1)
            th = torch.linalg.solve(A.T @ (wt[:, None] * A), A.T @ (wt * V))
            pred = (A @ th).clamp_min(1e-9)
            ss_res = (wt * (V - pred) ** 2).sum()
            ss_tot = (wt * (V - (wt * V).sum()) ** 2).sum()
            lr = torch.log(V) - torch.log(pred)
            lres = (wt * lr ** 2).sum()
            ltot = (wt * (torch.log(V) - (wt * torch.log(V)).sum()) ** 2).sum()
            return th, float(1 - ss_res / ss_tot), float(1 - lres / ltot)

        th_lin, r2_lin, lr2_lin = wls([torch.ones_like(c), c])
        th_q, r2_q, lr2_q = wls([torch.ones_like(c), c, c ** 2])
        self.p = (float(th_q[0]), float(th_q[1]), float(th_q[2]))
        return {"model": "sigma^2 = b + a*mu + c*mu^2 (Poisson-Gaussian + multiplicative)",
                "b": self.p[0], "a": self.p[1], "c": self.p[2],
                "R2_var": r2_q, "R2_logvar": lr2_q,
                "linear_poisson_gaussian_only": {"b": float(th_lin[0]), "a": float(th_lin[1]),
                                                  "R2_var": r2_lin, "R2_logvar": lr2_lin},
                "bin_centers": [float(x) for x in c], "measured_var": [float(x) for x in V],
                "bin_weight": [float(x) for x in wt]}

    @torch.no_grad()
    def mean_residual(self, X_u8, bs=32):
        dev = self.c.device
        tot, n = 0.0, 0
        for i in range(0, X_u8.shape[0], bs):
            I = X_u8[i:i + bs].to(dev).float().div(255).unsqueeze(1)
            tot += float(self.residual(I).sum())
            n += I.shape[0]
        return tot / max(n, 1)

    def state(self):
        return {"p": list(self.p), "k": self.k, "K": self.K, "lo": self.lo, "hi": self.hi, "mask_lo": self.mask_lo}

    def load(self, s):
        self.p = tuple(s["p"])
''',
         "a6_classifier.py": r'''# a6_classifier.py - A6: real-data benign/malignant classifier on the CLEAN split (+ shared training helpers).
# Used as (i) the "real-only" reference of the later downstream test, (ii) the judge of class conditioning (A7/A9),
# (iii) the "mammography feature" extractor for KID-mammo.
# PRE-STATED RECIPE (no tuning, no epoch selection): torchvision ConvNeXt-Tiny with ImageNet-1k weights,
# 256 px grayscale -> 3 channels, per-image min-max, ImageNet normalisation, AdamW lr 1e-4 wd 0.05, batch 32,
# 15 epochs, cosine schedule with 1 warm-up epoch, horizontal flip + small affine, class-balanced cross-entropy,
# fp16 autocast. FINAL epoch is used. Seeds 0, 1, 2. Trained on clean TRAIN minus the patient-level VALIDATION split.
# PRE-STATED RULE (validation only - the test set is never used for a decision):
#   usable as judge if the mean VALIDATION AUC over the 3 seeds is >= 0.65.  Test AUC is reported, not used.
import os, sys, time, json, math, argparse
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from phaseA_common import log, save_json, load_cache, load_clean_meta, val_mask, OUT

TAG = "A6"
dev = torch.device("cuda")
torch.backends.cudnn.benchmark = True
MEAN = torch.tensor([0.485, 0.456, 0.406], device=dev).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225], device=dev).view(1, 3, 1, 1)


def prep(x01):
    """(B,1,H,W) any range -> per-image min-max -> 3ch ImageNet-normalised"""
    mn = x01.amin(dim=(1, 2, 3), keepdim=True); mx = x01.amax(dim=(1, 2, 3), keepdim=True)
    x = (x01 - mn) / (mx - mn + 1e-8)
    return (x.repeat(1, 3, 1, 1) - MEAN) / STD


def build():
    import torchvision
    m = torchvision.models.convnext_tiny(weights="IMAGENET1K_V1")
    m.classifier[2] = nn.Linear(m.classifier[2].in_features, 2)
    return m


def auc(scores, labels):
    """Mann-Whitney AUC with tie handling (no sklearn dependency)"""
    s, y = np.asarray(scores, float), np.asarray(labels, int)
    order = np.argsort(s); r = np.empty(len(s)); r[order] = np.arange(1, len(s) + 1)
    for v in np.unique(s):
        m = s == v
        if m.sum() > 1:
            r[m] = r[m].mean()
    n1, n0 = (y == 1).sum(), (y == 0).sum()
    return float((r[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0))


@torch.no_grad()
def predict(m, X_u8, bs=64):
    m.eval(); out = []
    for i in range(0, len(X_u8), bs):
        xb = torch.as_tensor(X_u8[i:i + bs]).to(dev).float().div(255).unsqueeze(1)
        with torch.autocast("cuda", dtype=torch.float16):
            out.append(torch.softmax(m(prep(xb)).float(), 1)[:, 1].cpu())
    return torch.cat(out).numpy()


def augment(x, g):
    B = x.shape[0]
    u = lambda: torch.rand(B, generator=g, device=dev) * 2 - 1
    sx, sy, shx, shy = 1 + 0.05 * u(), 1 + 0.05 * u(), 0.05 * u(), 0.05 * u()
    z = torch.zeros(B, device=dev)
    th = torch.stack([torch.stack([sx, shx, z], 1), torch.stack([shy, sy, z], 1)], 1)
    x = F.grid_sample(x, F.affine_grid(th, list(x.shape), align_corners=False), padding_mode="zeros", align_corners=False)
    flip = torch.rand(B, generator=g, device=dev) < 0.5
    return torch.where(flip.view(-1, 1, 1, 1), x.flip(-1), x)


def train_clf(X_u8, y, seed, epochs=15, bs=32, tag=TAG):
    """the fixed recipe above; returns the model after the FINAL epoch"""
    t0 = time.time()
    Xg, Yg = torch.from_numpy(X_u8).to(dev), torch.from_numpy(np.asarray(y)).long().to(dev)
    w = torch.tensor([len(y) / (2 * max(1, (np.asarray(y) == c).sum())) for c in (0, 1)], device=dev, dtype=torch.float32)
    torch.manual_seed(seed)
    m = build().to(dev)
    opt = torch.optim.AdamW(m.parameters(), lr=1e-4, weight_decay=0.05)
    spe = math.ceil(len(y) / bs); total = spe * epochs
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0, (s + 1) / spe) * 0.5 * (1 + math.cos(math.pi * s / total)))
    scaler = torch.amp.GradScaler("cuda")
    g = torch.Generator(device=dev).manual_seed(seed)
    for ep in range(epochs):
        m.train()
        perm = torch.randperm(len(y), generator=torch.Generator().manual_seed(seed * 100 + ep)).to(dev)
        tl = 0.0
        for i in range(spe):
            idx = perm[i * bs:(i + 1) * bs]
            xb = augment(Xg[idx].float().div(255).unsqueeze(1), g)
            with torch.autocast("cuda", dtype=torch.float16):
                loss = F.cross_entropy(m(prep(xb)).float(), Yg[idx], weight=w)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
            tl += float(loss)
        if (ep + 1) % 5 == 0 or ep == 0:
            log(tag, f"seed {seed} epoch {ep + 1}/{epochs} train loss {tl / spe:.4f} | {(time.time() - t0) / 60:.1f} min")
    del Xg, Yg
    return m


def boot_auc(p, y, n=1000, seed=0):
    r = np.random.default_rng(seed); vals = []
    i0, i1 = np.where(np.asarray(y) == 0)[0], np.where(np.asarray(y) == 1)[0]
    for _ in range(n):
        b = np.concatenate([r.choice(i0, len(i0)), r.choice(i1, len(i1))])
        vals.append(auc(p[b], np.asarray(y)[b]))
    return [float(np.quantile(vals, 0.025)), float(np.quantile(vals, 0.975))]


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--seeds", default="0,1,2")
    ap.add_argument("--epochs", type=int, default=15)
    a = ap.parse_args()
    x, y, pat, _ = load_clean_meta("train"); xte, yte = load_cache("test")
    vm = val_mask(pat)
    res = {"recipe": "convnext_tiny IMAGENET1K_V1, AdamW 1e-4 wd .05, bs 32, 15 ep cosine (1 warm-up), flip+affine, "
                     "balanced CE, final epoch; trained on train minus validation patients",
           "n_train_fit": int((~vm).sum()), "n_val": int(vm.sum()), "n_val_patients": len(set(pat[vm])),
           "n_test": int(len(yte)), "seeds": {}}
    log(TAG, f"fit on {res['n_train_fit']} images | validation {res['n_val']} images ({res['n_val_patients']} patients)")
    pv_sum = np.zeros(int(vm.sum()))
    for seed in [int(s) for s in a.seeds.split(",")]:
        t0 = time.time()
        m = train_clf(x[~vm], y[~vm], seed, a.epochs)
        pv, pt = predict(m, x[vm]), predict(m, xte)
        pv_sum += pv
        res["seeds"][str(seed)] = {"val_auc": auc(pv, y[vm]), "test_auc_reported_only": auc(pt, yte),
                                   "val_balanced_acc": float(np.mean([((pv >= 0.5) == (y[vm] == 1))[y[vm] == c].mean() for c in (0, 1)])),
                                   "minutes": (time.time() - t0) / 60}
        torch.save({k: v.half() for k, v in m.state_dict().items()}, os.path.join(OUT, f"a6_clf_s{seed}.pt"))
        log(TAG, f"seed {seed}: VALIDATION AUC {res['seeds'][str(seed)]['val_auc']:.3f} | test AUC (reported only) "
                 f"{res['seeds'][str(seed)]['test_auc_reported_only']:.3f} | {(time.time() - t0) / 60:.1f} min")
        del m; torch.cuda.empty_cache()
    va = [v["val_auc"] for v in res["seeds"].values()]
    res["mean_val_auc"] = float(np.mean(va)); res["sd_val_auc"] = float(np.std(va, ddof=1)) if len(va) > 1 else 0.0
    pe = pv_sum / len(va)
    res["ensemble_val_auc"] = auc(pe, y[vm]); res["ensemble_val_auc_ci95"] = boot_auc(pe, y[vm])
    res["mean_test_auc_reported_only"] = float(np.mean([v["test_auc_reported_only"] for v in res["seeds"].values()]))
    res["usable_as_judge"] = bool(res["mean_val_auc"] >= 0.65)
    save_json(res, "A6_classifier.json")
    log(TAG, f"mean VALIDATION AUC {res['mean_val_auc']:.3f} +- {res['sd_val_auc']:.3f} (ensemble {res['ensemble_val_auc']:.3f}) "
             f"-> usable as judge: {res['usable_as_judge']}")
''',
         "train_ema.py": r'''# train_ema.py (S10) = train_arm2.py + EMA of the weights (see S10_preregistration.txt).
# train_arm2.py - Experiment 2: PINN ablation on the ORIGINAL training recipe (monai_ddpm.py / train.py).
# Same as the original: 0.7*MSE + 0.2*(1-MS-SSIM) + 0.05*HF loss on x0_hat, per-image min-max to [-1,1],
# shear/scale/flip augmentation, 15% label dropout (-1), batch 2 x grad-accum 4, Adam, clip 1.0, fp16.
# The ONLY difference between arms: PINN adds lam * physics-consistency hinge term.
import os, sys, glob, time, json, argparse
def _walk_find(name, root="/kaggle/input"):
    """S8: like glob('/kaggle/input/**/name') but skips the huge image folders (CMMD DICOM tree, CBIS jpeg)"""
    out = []
    for dp, dns, fns in os.walk(root):
        dns[:] = [d for d in dns if d not in ("CMMD", "jpeg", "U24_synthesis", ".git")]
        if name in fns:
            out.append(os.path.join(dp, name))
    return sorted(out)

import numpy as np
import torch
import torch.nn.functional as F
from torchmetrics.functional.image import multiscale_structural_similarity_index_measure as msssim

WD = "/kaggle/working/U24_synthesis/lightning_synthesis"
sys.path.insert(0, WD)
from pinn_physics import PhysicsNLF

ap = argparse.ArgumentParser()
ap.add_argument("--arm", required=True)                  # base2 | pinn2
ap.add_argument("--lam", type=float, default=0.0)        # physics weight, 0 = baseline
ap.add_argument("--epochs", type=int, default=60)
ap.add_argument("--bs", type=int, default=2)
ap.add_argument("--accum", type=int, default=4)
ap.add_argument("--lr", type=float, default=1e-5)
ap.add_argument("--p_uncond", type=float, default=0.15)
ap.add_argument("--tau_q", type=float, default=0.9)      # hinge threshold = this quantile of REAL residuals
ap.add_argument("--seed", type=int, default=42)
ap.add_argument("--budget_h", type=float, default=10.9)
ap.add_argument("--snap_every", type=int, default=5)
ap.add_argument("--snap_epochs", default="")            # S10: comma list of epochs that get RAW + EMA snapshots
ap.add_argument("--ema_decay", type=float, default=0.999)  # S10: per-optimizer-step decay
ap.add_argument("--ema_every", type=int, default=2)        # S10: EMA updated every k steps with decay**k (CPU, fp32)
ap.add_argument("--smoke_batches", type=int, default=0)
ap.add_argument("--log_every", type=int, default=200)
ap.add_argument("--ckpt_dir", default="/kaggle/working/ckpt")
a = ap.parse_args()

TAG = f"[{a.arm}]"
def log(*m):
    print(TAG, *m, flush=True)

try:
    T0 = float(open("/tmp/pg_session_start").read())
except Exception:
    T0 = time.time()
dev = torch.device("cuda")
torch.backends.cudnn.benchmark = True
os.makedirs(a.ckpt_dir, exist_ok=True)

def save_atomic(obj, path):
    tmp = path + ".tmp"
    torch.save(obj, tmp)
    with open(tmp, "rb") as fh:
        os.fsync(fh.fileno())
    os.replace(tmp, path)

def minmax_pm1(x):                                   # per-image min-max -> [-1,1] (= original z-score + ScaleIntensity)
    mn = x.amin(dim=(1, 2, 3), keepdim=True)
    mx = x.amax(dim=(1, 2, 3), keepdim=True)
    return 2 * (x - mn) / (mx - mn + 1e-8) - 1

# ---------------- original HighFreqLoss (copied from monai_ddpm.py) ----------------
_sob = torch.tensor([[1, 0, -1], [2, 0, -2], [1, 0, -1]], dtype=torch.float32)
KX = (_sob[None, None] / 8).to(dev)
KY = (_sob.t()[None, None] / 8).to(dev)
def hf_loss(x, y):
    gx = F.conv2d(x, KX, padding=1) - F.conv2d(y, KX, padding=1)
    gy = F.conv2d(x, KY, padding=1) - F.conv2d(y, KY, padding=1)
    return (gx.pow(2) + gy.pow(2)).mean()

# ---------------- data (uint8 cache, kept on GPU) ----------------
d = np.load(os.path.join(os.environ.get("PA_TMP", "/tmp"), "train_cache.npz"))   # S8: dataset switch (default = CBIS-DDSM)
X = torch.from_numpy(d["x"])
Y = torch.from_numpy(d["y"]).long()
N = X.shape[0]
Xg, Yg = X.to(dev), Y.to(dev)
log(f"data {N} images {tuple(X.shape[1:])} benign={int((Y == 0).sum())} malignant={int((Y == 1).sum())} "
    f"on {torch.cuda.get_device_name(0)}")

def augment(img, gg):
    # img: (B,1,H,W) float in 0..255. Effective original aug: shear +-0.1, scale +-5% (p=0.9), h-flip p=0.5
    B = img.shape[0]
    do = (torch.rand(B, generator=gg, device=dev) < 0.9).float()
    u = lambda: (torch.rand(B, generator=gg, device=dev) * 2 - 1)
    sx, sy, shx, shy = 1 + 0.05 * u() * do, 1 + 0.05 * u() * do, 0.1 * u() * do, 0.1 * u() * do
    zero = torch.zeros(B, device=dev)
    theta = torch.stack([torch.stack([sx, shx, zero], 1), torch.stack([shy, sy, zero], 1)], 1)
    grid = F.affine_grid(theta, list(img.shape), align_corners=False)
    out = F.grid_sample(img, grid, mode="bilinear", padding_mode="zeros", align_corners=False)
    flip = torch.rand(B, generator=gg, device=dev) < 0.5
    return torch.where(flip.view(-1, 1, 1, 1), out.flip(-1), out)

# ---------------- model: the repo's exact DiffusionModelUNet + noise schedule ----------------
from model_architectures import MonaiDDPM
torch.manual_seed(a.seed)                       # identical initial weights for both arms
_m = MonaiDDPM(lr=a.lr, T=1000)
unet = _m.unet.to(dev)
ac = _m.scheduler.alphas_cumprod.detach().float().to(dev)
del _m
T = ac.numel()
opt = torch.optim.Adam(unet.parameters(), lr=a.lr)
scaler = torch.amp.GradScaler("cuda")
nlf = PhysicsNLF(dev)
tau = None

# ---------------- resume ----------------
paths = [os.path.join(a.ckpt_dir, f"{a.arm}_last.pt")]
if not a.smoke_batches:
    paths += _walk_find(f"{a.arm}_last.pt")
cands = []
for p in paths:
    if os.path.exists(p):
        try:
            ck = torch.load(p, map_location="cpu", weights_only=False, mmap=True)
            cands.append((int(ck["step"]), p))
            del ck
        except Exception as e:
            log("unreadable checkpoint (skipped):", p, repr(e)[:150])
start_epoch, step, opt_steps, history, resumed = 0, 0, 0, [], None
if cands:
    resumed = max(cands)[1]
    ck = torch.load(resumed, map_location="cpu", weights_only=False)
    for k in ("lam", "lr", "bs", "accum", "seed", "p_uncond", "tau_q"):
        if ck["args"].get(k) != getattr(a, k):
            log(f"!! WARNING {k} differs: checkpoint={ck['args'].get(k)} now={getattr(a, k)}")
    unet.load_state_dict(ck["unet"])
    opt.load_state_dict(ck["opt"])
    scaler.load_state_dict(ck["scaler"])
    start_epoch, step, opt_steps, history = int(ck["epoch"]), int(ck["step"]), int(ck["opt_steps"]), ck["history"]
    nlf.load(ck["nlf"])
    tau = float(ck["tau"])
    del ck
    torch.cuda.empty_cache()

# ---------------- physics calibration on REAL images, in the training intensity domain ----------------
Xf = X.float()
mn = Xf.amin(dim=(1, 2), keepdim=True); mx = Xf.amax(dim=(1, 2), keepdim=True)
Xmm = ((Xf - mn) / (mx - mn + 1e-8) * 255).round().clamp(0, 255).to(torch.uint8)
del Xf
if nlf.p is None:
    rep = nlf.fit(Xmm)
    log(f"NLF fit: sigma^2 = {rep['b']:.3e} + {rep['a']:.3e}*mu + {rep['c']:.3e}*mu^2 | "
        f"R2(var)={rep['R2_var']:.3f} R2(logvar)={rep['R2_logvar']:.3f}")
with torch.no_grad():
    res_real = torch.cat([nlf.residual(Xmm[i:i + 32].to(dev).float().div(255).unsqueeze(1)).cpu()
                          for i in range(0, N, 32)])
if tau is None:
    tau = float(torch.quantile(res_real, a.tau_q))
    with open(os.path.join(a.ckpt_dir, f"nlf_fit_{a.arm}.json"), "w") as fh:
        json.dump({**rep, "tau": tau, "tau_q": a.tau_q, "real_mean_residual": float(res_real.mean())}, fh, indent=2)
log(f"physics residual of REAL images: mean={float(res_real.mean()):.4f} | hinge threshold tau(q={a.tau_q})={tau:.4f}")

# ---------------- S10: EMA of the weights (fp32, on CPU to fit T4 memory; updated every ema_every optimizer steps) ----
PNAMES = [n for n, _ in unet.named_parameters()]; PLIST = [p for _, p in unet.named_parameters()]
EMA = None
if resumed:
    _ck = torch.load(resumed, map_location="cpu", weights_only=False)
    if "ema" in _ck:
        EMA = [_ck["ema"][n].float().clone() for n in PNAMES]
    del _ck
if EMA is None:
    EMA = [p.detach().float().cpu().clone() for p in PLIST]
    log(f"EMA initialised from the {'resumed' if resumed else 'initial'} weights")
else:
    log("EMA restored from checkpoint")
SNAP_EP = {int(v) for v in a.snap_epochs.split(",") if v.strip()}
def ema_update(n_steps):
    d = min(a.ema_decay, (1 + n_steps) / (10 + n_steps)) ** a.ema_every
    cur = [p.detach().float().cpu() for p in PLIST]
    torch._foreach_lerp_(EMA, cur, 1.0 - d)
def ema_state_dict():
    sd = {k: v.detach().half().cpu() for k, v in unet.state_dict().items()}
    for n, e in zip(PNAMES, EMA):
        sd[n] = e.half()
    return sd
n_opt = sum(1 for s in opt.state.values() if s)
chk = float(sum(p.detach().float().sum() for p in list(unet.parameters())[:5]))
log(f"START epoch={start_epoch} step={step} opt_states={n_opt} lam={a.lam} checksum={chk:.6f} "
    f"resumed={resumed or 'scratch'}")
if resumed and n_opt == 0:
    log("!! WARNING: checkpoint loaded but optimizer state is EMPTY -> not a true resume")

spe = N // a.bs
if a.smoke_batches:
    spe = min(spe, a.smoke_batches)
durs = [h["minutes"] * 60 for h in history[-3:]]
reason = "max epochs reached"
bad = 0
for epoch in range(start_epoch, a.epochs):
    used_h = (time.time() - T0) / 3600
    if durs and used_h + 1.15 * max(durs[-3:]) / 3600 + 0.2 > a.budget_h:
        reason = f"time budget ({used_h:.2f} h used of {a.budget_h} h)"
        break
    gcpu = torch.Generator().manual_seed(a.seed * 1000 + epoch)
    perm = torch.randperm(N, generator=gcpu)
    gg = torch.Generator(device=dev).manual_seed(a.seed * 1000 + epoch)
    unet.train()
    opt.zero_grad(set_to_none=True)
    t_ep, t_win = time.time(), time.time()
    S = dict(loss=0.0, comp=0.0, mse=0.0, ms=0.0, hf=0.0, phys=0.0, hinge=0.0)
    n_ok = 0
    for i in range(spe):
        idx = perm[i * a.bs:(i + 1) * a.bs].to(dev)
        x0 = minmax_pm1(augment(Xg[idx].float().unsqueeze(1), gg))
        y = Yg[idx]
        t = torch.randint(0, T, (a.bs,), generator=gg, device=dev)
        eps = torch.randn(x0.shape, generator=gg, device=dev)
        drop = torch.rand(a.bs, generator=gg, device=dev) < a.p_uncond
        cond = torch.where(drop, torch.full_like(y, -1), y).float().view(a.bs, 1, 1)
        ab = ac[t].view(-1, 1, 1, 1)
        xt = ab.sqrt() * x0 + (1 - ab).sqrt() * eps
        with torch.autocast("cuda", dtype=torch.float16):
            eps_hat = unet(xt, timesteps=t, context=cond)
        eps_hat = eps_hat.float()
        mse = F.mse_loss(eps_hat, eps)
        x0_hat = ((xt - (1 - ab).sqrt() * eps_hat) / ab.sqrt()).clamp(-1, 1)
        ms = 1.0 - msssim((x0_hat + 1) / 2, (x0 + 1) / 2, data_range=2.0, gaussian_kernel=False, normalize="relu")
        hf = hf_loss(x0_hat, x0)
        comp = 0.7 * mse + 0.2 * ms + 0.05 * hf                     # exact original loss
        w = ab.view(-1)
        if a.lam > 0:
            r = nlf.residual((x0_hat + 1) / 2)
            hinge = (w * F.relu(r - tau)).mean()
            loss = comp + a.lam * hinge
        else:
            with torch.no_grad():                                   # baseline: tracked, never trained on
                r = nlf.residual((x0_hat + 1) / 2)
                hinge = (w * F.relu(r - tau)).mean()
            loss = comp
        phys = (w * r).mean().detach()
        if not torch.isfinite(loss):
            bad += 1
            if bad > 50:
                raise RuntimeError("50 consecutive non-finite losses - stopping")
            continue
        bad = 0
        scaler.scale(loss / a.accum).backward()
        step += 1
        if (i + 1) % a.accum == 0 or i == spe - 1:
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(unet.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()
            opt.zero_grad(set_to_none=True)
            opt_steps += 1
            if opt_steps % a.ema_every == 0:
                ema_update(opt_steps)
        n_ok += 1
        for k, v in (("loss", loss), ("comp", comp), ("mse", mse), ("ms", ms), ("hf", hf), ("phys", phys), ("hinge", hinge)):
            S[k] += float(v.detach())
        if (i + 1) % a.log_every == 0:
            ips = a.log_every / (time.time() - t_win)
            t_win = time.time()
            m = {k: v / n_ok for k, v in S.items()}
            log(f"ep {epoch + 1} it {i + 1}/{spe} step {step} loss {m['loss']:.4f} comp {m['comp']:.4f} "
                f"mse {m['mse']:.4f} ms {m['ms']:.4f} hf {m['hf']:.4f} phys {m['phys']:.4f} "
                f"hinge {m['hinge']:.4f} | {ips:.2f} it/s")
    dur = time.time() - t_ep
    durs.append(dur)
    m = {k: v / max(n_ok, 1) for k, v in S.items()}
    history.append({"epoch": epoch + 1, "step": step, "opt_steps": opt_steps, **m, "minutes": dur / 60})
    save_atomic({"unet": unet.state_dict(), "opt": opt.state_dict(), "scaler": scaler.state_dict(),
                 "epoch": epoch + 1, "step": step, "opt_steps": opt_steps, "args": vars(a), "nlf": nlf.state(),
                 "tau": tau, "history": history, "ema": {n: e for n, e in zip(PNAMES, EMA)}}, os.path.join(a.ckpt_dir, f"{a.arm}_last.pt"))
    snap = {"unet": {k: v.detach().half().cpu() for k, v in unet.state_dict().items()},
            "epoch": epoch + 1, "step": step, "arm": a.arm, "lam": a.lam, "nlf": nlf.state(), "tau": tau}
    save_atomic(snap, os.path.join(a.ckpt_dir, f"{a.arm}_w_latest.pt"))
    if (epoch + 1) % a.snap_every == 0 or (epoch + 1) in SNAP_EP:
        save_atomic(snap, os.path.join(a.ckpt_dir, f"{a.arm}_w_ep{epoch + 1:03d}.pt"))
    if (epoch + 1) in SNAP_EP:
        save_atomic({**snap, "unet": ema_state_dict(), "ema": True}, os.path.join(a.ckpt_dir, f"{a.arm}_ema_ep{epoch + 1:03d}.pt"))
    del snap
    with open(os.path.join(a.ckpt_dir, f"{a.arm}_history.json"), "w") as fh:
        json.dump(history, fh, indent=1)
    left = a.epochs - epoch - 1
    log(f"SAVED epoch={epoch + 1} step={step} opt_steps={opt_steps} loss={m['loss']:.4f} comp={m['comp']:.4f} "
        f"mse={m['mse']:.4f} ms={m['ms']:.4f} hf={m['hf']:.4f} phys={m['phys']:.4f} hinge={m['hinge']:.4f} | "
        f"{dur / 60:.1f} min/epoch | peak GPU mem {torch.cuda.max_memory_allocated() / 1e9:.1f} GB | "
        f"~{left * dur / 3600:.1f} h for {left} more")
log(f"FINISHED epochs_completed={history[-1]['epoch'] if history else start_epoch} step={step} reason={reason}")
''',
         "s6_generate.py": r'''# s6_generate.py - S6 sample generation (= s5_generate.py + optional dynamic thresholding) (no training). Same DDIM-50 + classifier-free guidance sampler and the SAME
# noise bank as s3_eval / s4_eval (noise seed 2025 + 100000*label + batch, batch 16, benign first), so image i of every
# set starts from the same noise and label (paired comparisons between checkpoints / sampler settings).
#   --ckpts a.pt[,b.pt,...]  one snapshot, or several -> their weights are averaged (uniform checkpoint averaging)
#   --eps_scale s            Epsilon Scaling (Ning et al., ICLR 2024): the guided noise prediction is divided by s at
#                            every step (s = 1: unchanged sampler)
#   --dt p                   dynamic thresholding (Saharia et al. 2022, Imagen): at every step the predicted clean image
#                            x0 is clipped to [-s, s] and divided by s, s = max(1, p-th percentile of |x0| per image);
#                            p = 0 -> the original fixed clamp to [-1, 1]
import os, sys, time, json, argparse
import numpy as np, torch
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from phaseA_common import log, save_json, schedule, OUT

ap = argparse.ArgumentParser()
ap.add_argument("--name", required=True)
ap.add_argument("--ckpts", required=True)
ap.add_argument("--n_per_class", type=int, default=100)
ap.add_argument("--gs", type=float, default=2.0)
ap.add_argument("--eps_scale", type=float, default=1.0)
ap.add_argument("--steps", type=int, default=50)
ap.add_argument("--noise_seed", type=int, default=2025)
ap.add_argument("--dt", type=float, default=0.0)
a = ap.parse_args()
TAG = f"S6-gen-{a.name}"
dev = torch.device("cuda")
paths = a.ckpts.split(",")
unet, ac = schedule(dev)
avg = None
for p in paths:
    sd = torch.load(p, map_location="cpu", weights_only=False)
    sd = sd["unet"] if "unet" in sd else sd["models"]["net"]
    if avg is None:
        avg = {k: v.float().clone() for k, v in sd.items()}
    else:
        for k in avg:
            avg[k] += sd[k].float()
    del sd
for k in avg:
    avg[k] /= len(paths)
unet.load_state_dict(avg); del avg
unet = unet.to(dev).half().eval()
for q in unet.parameters():
    q.requires_grad_(False)
acf = ac.float(); T = acf.numel()
log(TAG, f"{len(paths)} checkpoint(s) {'(weights averaged)' if len(paths) > 1 else ''}: {paths} | {a.n_per_class}/class | gs {a.gs} | "
         f"eps_scale {a.eps_scale} | dynamic threshold {a.dt or 'off'} | noise seed {a.noise_seed} | {a.steps} DDIM steps")

@torch.no_grad()
def ddim(label, n, gen):
    x = torch.randn(n, 1, 256, 256, generator=gen, device=dev)
    ts = torch.linspace(T - 1, 0, a.steps, device=dev).round().long()
    cond = torch.full((n, 1, 1), float(label), device=dev)
    for i, t in enumerate(ts):
        ab_t = acf[t]
        ab_prev = acf[ts[i + 1]] if i + 1 < len(ts) else torch.tensor(1.0, device=dev)
        tb = torch.full((2 * n,), int(t), device=dev, dtype=torch.long)
        ctx = torch.cat([-torch.ones_like(cond), cond]).half()
        eu, ec = unet(torch.cat([x, x]).half(), timesteps=tb, context=ctx).float().chunk(2)
        eps = (eu + a.gs * (ec - eu)) / a.eps_scale
        x0 = (x - (1 - ab_t).sqrt() * eps) / ab_t.sqrt()
        if a.dt > 0:
            sdt = torch.quantile(x0.abs().flatten(1), a.dt / 100.0, dim=1).clamp_min(1.0).view(-1, 1, 1, 1)
            x0 = torch.maximum(torch.minimum(x0, sdt), -sdt) / sdt
        else:
            x0 = x0.clamp(-1, 1)
        eps = (x - ab_t.sqrt() * x0) / (1 - ab_t).sqrt()
        x = ab_prev.sqrt() * x0 + (1 - ab_prev).sqrt() * eps
    return ((x + 1) / 2).clamp(0, 1)

t0 = time.time(); imgs, labels = [], []
for label in (0, 1):
    done, b = 0, 0
    while done < a.n_per_class:
        k = min(16, a.n_per_class - done)
        out = ddim(label, k, torch.Generator(device=dev).manual_seed(a.noise_seed + 100000 * label + b))[:, 0].cpu()
        imgs.append((out.numpy() * 255).round().astype(np.uint8)); labels += [label] * k; done += k; b += 1
    log(TAG, f"label {label}: {done} images | {(time.time() - t0) / 60:.1f} min")
G = np.concatenate(imgs); L = np.array(labels, dtype=np.int64)
np.savez_compressed(os.path.join(OUT, f"gen_{a.name}.npz"), x=G, y=L)
from PIL import Image
kk = min(8, a.n_per_class)
Image.fromarray(np.concatenate([np.concatenate([G[j + c * a.n_per_class] for j in range(kk)], 1) for c in (0, 1)], 0)).save(
    os.path.join(OUT, f"grid_{a.name}.png"))
save_json({"name": a.name, "ckpts": paths, "n_per_class": a.n_per_class, "gs": a.gs, "eps_scale": a.eps_scale, "dt": a.dt, "noise_seed": a.noise_seed, "steps": a.steps,
           "minutes": (time.time() - t0) / 60}, f"gen_{a.name}.json")
log(TAG, f"saved gen_{a.name}.npz ({len(G)} images) in {(time.time() - t0) / 60:.1f} min")
''',
         "s7_stats.py": r'''# s7_stats.py - copy of s6_stats.py with a selectable CI level (alpha). S6 statistics (pure numpy). Pre-stated rule (fixed 2026-10-21 ~17:10 Asia/Dhaka, before any S6 result):
#   For every arm vs the baseline arm (same noise bank -> generated image i is paired across arms):
#     KID-mammo  = unbiased MMD^2 (cubic kernel) between the generated set and the reference set in the A6 mammography-
#                  classifier feature space; KID-dino = the same in DINOv2 ViT-B/14 feature space (independent "eye")
#     precision / recall (Kynkaenniemi et al. 2019, k = 3) in DINOv2 space: realism / coverage (diversity)
#   PASS if ALL: KID-mammo <= 0.90 x baseline with the paired-bootstrap 95 % CI of the difference < 0;
#               KID-dino <= 1.00 x baseline (no loss in the independent feature space);
#               recall >= baseline recall - 0.05 (no loss of diversity).
#   Validation stage: reference = train-fit images; WINNER = passing arm with the lowest KID-mammo (none -> no winner).
#   Test stage (run ONCE, frozen winner vs baseline, fresh noise seeds): reference = the untouched TEST images; the same
#   rule -> CONFIRMED / NOT CONFIRMED.
import numpy as np
from scipy.spatial import cKDTree
from scipy.spatial.distance import cdist

kern = lambda A_, B_: (A_ @ B_.T / A_.shape[1] + 1.0) ** 3

def _kRR(R):
    K = kern(R, R); return (K.sum() - np.trace(K)) / (len(R) * (len(R) - 1))

def _mmd(KGG, h, kRR, c):
    num = c @ KGG @ c - (c * c * np.diag(KGG)).sum(); den = c.sum() ** 2 - (c * c).sum()
    return float(num / den - 2 * (c * h).sum() / c.sum() + kRR)

def prec_recall(real, fake, k=3):
    """improved precision / recall (Kynkaenniemi et al. 2019): k-NN balls around real (fake) points"""
    rr = cKDTree(real).query(real, k=k + 1)[0][:, -1]; rf = cKDTree(fake).query(fake, k=k + 1)[0][:, -1]
    D = cdist(fake, real)
    return float((D <= rr[None, :]).any(1).mean()), float((D.T <= rf[None, :]).any(1).mean())

def evaluate(F, base, n_boot=1000, seed=0, alpha=0.05):
    """F: {"ref_mammo", "ref_dino", "<arm>_mammo", "<arm>_dino", "<arm>_istats" (optional)}; returns (lines, results)"""
    arms = sorted({k[:-6] for k in F if k.endswith("_mammo") and not k.startswith("ref")})
    assert base in arms, f"baseline {base} missing"
    n = min(len(F[f"{a}_mammo"]) for a in arms)
    C = np.random.default_rng(seed).multinomial(n, np.ones(n) / n, size=n_boot).astype(float); ones = np.ones(n)
    res = {}
    for space in ("mammo", "dino"):
        R = F[f"ref_{space}"]; kRR = _kRR(R)
        for a in arms:
            G = F[f"{a}_{space}"][:n]; KGG, h = kern(G, G), kern(G, R).mean(1)
            res.setdefault(a, {})[f"kid_{space}"] = _mmd(KGG, h, kRR, ones)
            res[a][f"boot_{space}"] = np.array([_mmd(KGG, h, kRR, c) for c in C])
    for a in arms:
        res[a]["precision"], res[a]["recall"] = prec_recall(F["ref_dino"], F[f"{a}_dino"][:n])
        if f"{a}_istats" in F:
            st = F[f"{a}_istats"][:n]; res[a]["saturation"], res[a]["hf_energy"] = float(st[:, 1].mean()), float(st[:, 2].mean())
    b = res[base]; L = []; out = {}
    L.append(f"  baseline {base}: KID-mammo {b['kid_mammo']:.4f} | KID-dino {b['kid_dino']:.4f} | precision {b['precision']:.3f} "
             f"recall {b['recall']:.3f}" + (f" | saturation {b['saturation']:.4f} HF {b['hf_energy']:.4f}" if "saturation" in b else ""))
    for a in arms:
        r = res[a]
        if a == base:
            out[a] = {k: v for k, v in r.items() if not k.startswith("boot")}; continue
        dm = r["boot_mammo"] - b["boot_mammo"]; dd = r["boot_dino"] - b["boot_dino"]
        lo, hi = alpha / 2, 1 - alpha / 2
        ci_m = [float(np.quantile(dm, lo)), float(np.quantile(dm, hi))]; ci_d = [float(np.quantile(dd, lo)), float(np.quantile(dd, hi))]
        ok = {"KIDm<=0.90x": bool(r["kid_mammo"] <= 0.90 * b["kid_mammo"]), "KIDm_CI<0": bool(ci_m[1] < 0),
              "KIDd<=1.00x": bool(r["kid_dino"] <= b["kid_dino"]), "recall_ok": bool(r["recall"] >= b["recall"] - 0.05)}
        out[a] = {"kid_mammo": r["kid_mammo"], "kid_mammo_ratio": r["kid_mammo"] / b["kid_mammo"], "kid_mammo_diff_ci95": ci_m,
                  "kid_dino": r["kid_dino"], "kid_dino_ratio": r["kid_dino"] / b["kid_dino"], "kid_dino_diff_ci95": ci_d,
                  "precision": r["precision"], "recall": r["recall"], "saturation": r.get("saturation"), "hf_energy": r.get("hf_energy"),
                  "checks": ok, "passed": all(ok.values())}
        L.append(f"  {a:16s}: KID-mammo x{out[a]['kid_mammo_ratio']:.3f} (diff CI {ci_m[0]:+.4f}..{ci_m[1]:+.4f}) | KID-dino "
                 f"x{out[a]['kid_dino_ratio']:.3f} (CI {ci_d[0]:+.4f}..{ci_d[1]:+.4f}) | precision {r['precision']:.3f} recall {r['recall']:.3f}"
                 + (f" | saturation {r['saturation']:.4f} HF {r['hf_energy']:.4f}" if "saturation" in r else "")
                 + f" -> {'PASS' if out[a]['passed'] else 'fail'} {ok}")
    passed = sorted([(v["kid_mammo_ratio"], a) for a, v in out.items() if v.get("passed")])
    return L, out, (passed[0][1] if passed else None)

def _auc(p, y):
    from scipy.stats import rankdata
    r = rankdata(p); n1 = (y == 1).sum(); n0 = (y == 0).sum()
    return float((r[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0))

def downstream_summary(D, ref="real_only", n_boot=2000, seed=0):
    """D: {arm: {"y": y, "p0": ..., "p1": ..., ...}} -> lines (secondary, descriptive)"""
    L = []; rng = np.random.default_rng(seed)
    y = D[ref]["y"]; i0, i1 = np.where(y == 0)[0], np.where(y == 1)[0]
    B = [np.concatenate([rng.choice(i0, len(i0)), rng.choice(i1, len(i1))]) for _ in range(n_boot)]
    ens = {a: np.mean([v for k, v in d.items() if k.startswith("p")], 0) for a, d in D.items()}
    for a, d in D.items():
        aucs = [_auc(v, y) for k, v in d.items() if k.startswith("p")]
        line = f"  {a:16s}: TEST AUC {np.mean(aucs):.4f} +- {np.std(aucs, ddof=1) if len(aucs) > 1 else 0:.4f} (seeds) | ensemble {_auc(ens[a], y):.4f}"
        if a != ref:
            diff = [_auc(ens[a][b], y[b]) - _auc(ens[ref][b], y[b]) for b in B]
            line += f" | vs {ref}: {_auc(ens[a], y) - _auc(ens[ref], y):+.4f} (CI {np.quantile(diff, .025):+.4f}..{np.quantile(diff, .975):+.4f})"
        L.append(line)
    return L
''',
         "s10_eval.py": r'''# s10_eval.py - S10: s8_eval.py + the per-seed direction check (sets are 2 seeds concatenated: first half = seed 42,
# second half = seed 43). S8 evaluation = s7_eval.py (same features, same pre-stated rule from s7_stats.evaluate) plus
#   --clf    : the KID-mammo feature extractor of THIS dataset (CBIS: a6_clf_s0.pt from S2, CMMD: cmmd_clf_s0.pt)
#   Inception-v3 (torchvision ImageNet weights, pool 2048) FID and KID for every set - descriptive, not in the rule.
#   --stage val : reference = train-fit images of the dataset ($PA_DATA) | --stage test: reference = its TEST images ($PA_TMP)
import os, sys, json, argparse
import numpy as np, torch, torch.nn.functional as F
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from phaseA_common import log, save_json, load_clean_meta, load_cache, val_mask, OUT
from a6_classifier import build, prep
from s7_stats import evaluate, _kRR, _mmd, kern

ap = argparse.ArgumentParser()
ap.add_argument("--stage", choices=["val", "test"], required=True)
ap.add_argument("--sets", required=True)            # comma list, the FIRST is the reference arm
ap.add_argument("--clf", required=True)
ap.add_argument("--tag", required=True)
ap.add_argument("--alpha", type=float, default=0.05)
ap.add_argument("--n_boot", type=int, default=1000)
ap.add_argument("--halves", type=int, default=1)       # 1 = sets are two seeds concatenated -> per-seed direction check
a = ap.parse_args()
TAG = f"{a.tag}-{a.stage}"
dev = torch.device("cuda")
names = a.sets.split(","); base = names[0]
if a.stage == "val":
    xtr, ytr, pat, _ = load_clean_meta("train"); REF = xtr[~val_mask(pat)]
else:
    REF, _ = load_cache("test")
log(TAG, f"reference: {'train-fit' if a.stage == 'val' else 'TEST'} images {REF.shape} | sets {names} | classifier {a.clf}")

mam = build(); mam.load_state_dict({k: v.float() for k, v in torch.load(a.clf, map_location="cpu").items()}); mam = mam.to(dev).eval()
def load_dino():
    try:
        return torch.hub.load("facebookresearch/dinov2", "dinov2_vitb14", trust_repo=True)
    except Exception as e:
        log(TAG, f"torch.hub DINOv2 failed ({repr(e)[:150]}) -> timm")
        import timm
        return timm.create_model("vit_base_patch14_dinov2.lvd142m", pretrained=True, num_classes=0, img_size=224)
dino = load_dino().to(dev).eval()
import torchvision
inc = torchvision.models.inception_v3(weights="IMAGENET1K_V1"); inc.fc = torch.nn.Identity(); inc = inc.to(dev).eval()
MEAN = torch.tensor([0.485, 0.456, 0.406], device=dev).view(1, 3, 1, 1); STD = torch.tensor([0.229, 0.224, 0.225], device=dev).view(1, 3, 1, 1)
@torch.no_grad()
def feats(X_u8, bs=50):
    fm, fd, fi = [], [], []
    for i in range(0, len(X_u8), bs):
        xb = torch.from_numpy(X_u8[i:i + bs]).to(dev).float().div(255).unsqueeze(1)
        with torch.autocast("cuda", dtype=torch.float16):
            fm.append(mam.classifier[1](mam.classifier[0](mam.avgpool(mam.features(prep(xb))))).float().cpu())
        z = F.interpolate(xb, size=224, mode="bilinear", align_corners=False).repeat(1, 3, 1, 1)
        fd.append(dino((z - MEAN) / STD).float().cpu())
        z = F.interpolate(xb, size=299, mode="bilinear", align_corners=False).repeat(1, 3, 1, 1)
        fi.append(inc((z - MEAN) / STD).float().cpu())
    c = lambda L: torch.cat(L).numpy().astype(np.float64)
    return c(fm), c(fd), c(fi)
def istats_img(X_u8):
    I = torch.from_numpy(X_u8).float().div(255).unsqueeze(1)
    mn = I.amin(dim=(1, 2, 3), keepdim=True); mx = I.amax(dim=(1, 2, 3), keepdim=True); I = (I - mn) / (mx - mn + 1e-8)
    mu = F.avg_pool2d(I, 7, 1, 3, count_include_pad=False); m = (mu > 0.04).float(); n = m.sum((1, 2, 3)).clamp_min(1)
    hp = I - F.avg_pool2d(I, 7, 1, 3, count_include_pad=False)
    return np.stack([((I * m).sum((1, 2, 3)) / n).numpy(), ((((I > 0.95).float() * m).sum((1, 2, 3))) / n).numpy(),
                     ((hp * hp * m).sum((1, 2, 3)) / n).numpy(), m.mean((1, 2, 3)).numpy()], 1)
def fid(A, B):
    from scipy import linalg
    m1, m2 = A.mean(0), B.mean(0); c1, c2 = np.cov(A, rowvar=False), np.cov(B, rowvar=False)
    s, _ = linalg.sqrtm(c1 @ c2, disp=False)
    return float(((m1 - m2) ** 2).sum() + np.trace(c1 + c2 - 2 * np.real(s)))

Fd = {}
Fd["ref_mammo"], Fd["ref_dino"], Fd["ref_incep"] = feats(REF); Fd["ref_istats"] = istats_img(REF)
for nm in names:
    d = np.load(os.path.join(OUT, f"gen_{nm}.npz"))
    Fd[f"{nm}_mammo"], Fd[f"{nm}_dino"], Fd[f"{nm}_incep"] = feats(d["x"]); Fd[f"{nm}_istats"] = istats_img(d["x"])
np.savez(os.path.join(OUT, f"{a.tag}_{a.stage}_features.npz"), **Fd)
L, out, win = evaluate({k: v for k, v in Fd.items() if not k.endswith("_incep")}, base, a.n_boot, alpha=a.alpha)
kRR = _kRR(Fd["ref_incep"]); inc_lines = []
for nm in names:
    G = Fd[f"{nm}_incep"]
    kid = _mmd(kern(G, G), kern(G, Fd["ref_incep"]).mean(1), kRR, np.ones(len(G)))
    f_ = fid(G, Fd["ref_incep"]); out.setdefault(nm, {}).update({"kid_inception": kid, "fid_inception": f_})
    inc_lines.append(f"  {nm:16s}: Inception KID {kid:.5f} | Inception FID {f_:.2f}  (descriptive, n = {len(G)})")
ri = Fd["ref_istats"]
head = [f"[{a.tag} {a.stage.upper()}] CI level {100 * (1 - a.alpha):.2f} % | reference {'train-fit' if a.stage == 'val' else 'TEST'} "
        f"images (n = {len(REF)}): saturation {ri[:, 1].mean():.4f} HF {ri[:, 2].mean():.4f}"]
verdict = {n: bool(out[n].get("passed")) for n in names if n != base}
seed_lines = []
if a.halves:
    kRRm = _kRR(Fd["ref_mammo"])
    def kid_half(nm, h):
        G = Fd[f"{nm}_mammo"]; m = len(G) // 2; G = G[:m] if h == 0 else G[m:2 * m]
        return _mmd(kern(G, G), kern(G, Fd["ref_mammo"]).mean(1), kRRm, np.ones(len(G)))
    for n in names:
        if n == base:
            continue
        rat = [kid_half(n, h) / kid_half(base, h) for h in (0, 1)]
        same = all(r < 1.0 for r in rat)
        out[n]["seed_ratios_kid_mammo"] = rat; out[n]["same_direction_both_seeds"] = same
        verdict[n] = bool(verdict[n] and same)
        seed_lines.append(f"  {n:16s}: per-seed KID-mammo ratio seed42 x{rat[0]:.3f} | seed43 x{rat[1]:.3f} -> "
                          f"{'both < 1' if same else 'NOT both < 1'}")

tail = [f"  -> {n}: {'PASS' if v else 'fail'}" for n, v in verdict.items()]
save_json({"stage": a.stage, "alpha": a.alpha, "reference_arm": base, "verdict": verdict, "results": out}, f"{a.tag}_{a.stage}.json")
txt = "\n".join(head + L + ["  per-seed direction check (pre-registered, part of PASS):"] + seed_lines + ["  Inception-v3 (torchvision, descriptive):"] + inc_lines + tail)
open(os.path.join(OUT, f"{a.tag}_{a.stage}_report.txt"), "w").write(txt)
print(txt, flush=True)
''',
         "s10_pool.py": r'''# s10_pool.py - concatenate the seed-42 and seed-43 image sets of one arm (same noise bank -> image i stays paired)
import os, sys, numpy as np
OUT = os.environ["PA_OUT"]; out, a, b = sys.argv[1:4]
A, B = np.load(os.path.join(OUT, f"gen_{a}.npz")), np.load(os.path.join(OUT, f"gen_{b}.npz"))
assert (A["y"] == B["y"]).all(), "label order differs"
np.savez_compressed(os.path.join(OUT, f"gen_{out}.npz"), x=np.concatenate([A["x"], B["x"]]), y=np.concatenate([A["y"], B["y"]]))
print(f"[S10-pool] gen_{out}.npz = {a} + {b} ({len(A['y']) + len(B['y'])} images)", flush=True)
'''}
for n, s in FILES.items():
    Path(n).write_text(s)
Path("/tmp/prepare_cmmd.py").write_text(r'''# prepare_cmmd.py - CMMD cache for S8 (rules in S8_preregistration.txt, fixed before any CMMD model was trained).
#   cohort D1 only (D2 is malignant-only -> class would be confounded with cohort)
#   label = clinical-table "classification" joined per (patient ID, breast side) with the DICOM ImageLaterality tag
#   8-bit grey (MONOCHROME1 inverted), PIL resize to 256 x 256 (same call as for CBIS-DDSM)
#   patient-level split, seed 0, 80 % train / 20 % test, stratified by patient label
# Writes $PA_TMP/{train,test}_cache.npz (x, y) and $PA_DATA/{train,test}_cache.npz (x, y, md5, patient) + cmmd_summary.json.
# Reuses an earlier session's cmmd_data/ if attached (identical bytes).
import os, glob, json, shutil, hashlib, time, collections
def _walk_find(name, root="/kaggle/input"):
    """S8: like glob('/kaggle/input/**/name') but skips the huge image folders (CMMD DICOM tree, CBIS jpeg)"""
    out = []
    for dp, dns, fns in os.walk(root):
        dns[:] = [d for d in dns if d not in ("CMMD", "jpeg", "U24_synthesis", ".git")]
        if name in fns:
            out.append(os.path.join(dp, name))
    return sorted(out)

import numpy as np, pandas as pd
from PIL import Image

TMP = os.environ.get("PA_TMP", "/tmp/cmmd"); DATA = os.environ.get("PA_DATA", "/kaggle/working/cmmd_data")
os.makedirs(TMP, exist_ok=True); os.makedirs(DATA, exist_ok=True)

def to_tmp(src):
    for s in ("train", "test"):
        d = np.load(os.path.join(src, f"{s}_cache.npz"))
        np.savez(os.path.join(TMP, f"{s}_cache.npz"), x=d["x"], y=d["y"])
        print(f"CMMD {s}: {d['x'].shape} benign={int((d['y'] == 0).sum())} malignant={int((d['y'] == 1).sum())}")

if os.path.exists(os.path.join(DATA, "cmmd_summary.json")):
    to_tmp(DATA); print("CMMD cache already built in this session"); raise SystemExit(0)
prev = [p for p in _walk_find("cmmd_summary.json") if os.path.basename(os.path.dirname(p)) == "cmmd_data"]
if prev:
    for f in os.listdir(os.path.dirname(prev[0])):
        shutil.copy(os.path.join(os.path.dirname(prev[0]), f), os.path.join(DATA, f))
    to_tmp(DATA); print("CMMD cache REUSED from", os.path.dirname(prev[0])); raise SystemExit(0)

t0 = time.time()
xl = _walk_find("CMMD_clinicaldata_revision.xlsx")
if not xl:
    raise SystemExit("CMMD not attached: Add Input -> search 'CMMD' -> 'The Chinese Mammography Database (CMMD) 2022' (tommyngx)")
ROOT = os.path.dirname(xl[0])
clin = pd.read_excel(xl[0])
meta = pd.read_csv(os.path.join(ROOT, "metadata.csv"))
clin = clin[clin["ID1"].astype(str).str.startswith("D1-")].copy()
clin["side"] = clin["LeftRight"].astype(str).str.strip().str.upper()
clin["label"] = clin["classification"].astype(str).str.strip().str.lower().map({"benign": 0, "malignant": 1})
dup = clin.groupby(["ID1", "side"]).label.nunique()
amb = set(dup[dup > 1].index)                                   # same breast listed with two labels -> excluded
LAB = {k: int(v) for k, v in clin.groupby(["ID1", "side"]).label.first().items() if k not in amb and not pd.isna(v)}
print(f"clinical table: {len(clin)} D1 rows -> {len(LAB)} labelled breasts ({len(amb)} ambiguous excluded)", flush=True)

import pydicom
meta = meta[meta["Subject ID"].astype(str).str.startswith("D1-")]
rows, why = [], collections.Counter()
for _, r in meta.iterrows():
    folder = os.path.join(ROOT, str(r["File Location"]).replace("\\", "/").lstrip("./"))
    if not os.path.isdir(folder):
        why["series folder missing"] += 1; continue
    for f in sorted(os.listdir(folder)):
        if not f.lower().endswith(".dcm"):
            continue
        p = os.path.join(folder, f)
        ds = pydicom.dcmread(p)
        side = str(getattr(ds, "ImageLaterality", "") or getattr(ds, "Laterality", "")).strip().upper()
        pid = str(r["Subject ID"])
        if (pid, side) not in LAB:
            why["no unique label for this breast"] += 1; continue
        a = ds.pixel_array.astype(np.float64)
        if str(getattr(ds, "PhotometricInterpretation", "")).upper() == "MONOCHROME1":
            a = a.max() - a
        if ds.pixel_array.dtype != np.uint8:
            a = (a - a.min()) / (a.max() - a.min() + 1e-8) * 255
        im = Image.fromarray(np.clip(np.round(a), 0, 255).astype(np.uint8))
        x = np.array(im.resize((256, 256)), dtype=np.uint8)
        rows.append({"patient": pid, "side": side, "label": LAB[(pid, side)], "file": p,
                     "md5": hashlib.md5(x.tobytes()).hexdigest(), "x": x})
        why["used"] += 1
    if len(rows) and len(rows) % 400 == 0:
        print(f"  {len(rows)} images ({(time.time() - t0) / 60:.1f} min)", flush=True)
print("image resolution:", dict(why), flush=True)
D = pd.DataFrame([{k: v for k, v in r.items() if k != "x"} for r in rows])
X = np.stack([r["x"] for r in rows])
D = D.drop_duplicates("md5"); X = X[D.index.to_numpy()]; D = D.reset_index(drop=True)

# patient-level split, seed 0, stratified by patient label (a patient with any malignant breast = malignant)
plab = D.groupby("patient").label.max()
rng = np.random.default_rng(0); test_p = set()
for c in (0, 1):
    ps = sorted(plab[plab == c].index)
    test_p |= set(rng.choice(ps, int(round(0.2 * len(ps))), replace=False).tolist())
D["split"] = np.where(D.patient.isin(test_p), "test", "train")
assert not (set(D[D.split == "train"].patient) & set(D[D.split == "test"].patient))
summ = {"rule": "CMMD cohort D1; label per (patient, side) via DICOM ImageLaterality; 256x256; patient split 80/20 seed 0",
        "image_resolution": dict(why), "ambiguous_breasts_excluded": len(amb), "duplicates_removed": int(len(rows) - len(D))}
for s in ("train", "test"):
    m = (D.split == s).to_numpy(); d = D[m]
    x, y = X[m], d.label.to_numpy().astype(np.int64)
    np.savez(os.path.join(TMP, f"{s}_cache.npz"), x=x, y=y)
    np.savez_compressed(os.path.join(DATA, f"{s}_cache.npz"), x=x, y=y, md5=d.md5.to_numpy().astype(str),
                        patient=d.patient.to_numpy().astype(str))
    d[["patient", "side", "label", "md5", "file"]].to_csv(os.path.join(DATA, f"manifest_{s}.csv"), index=False)
    summ[s] = {"images": int(len(d)), "benign": int((y == 0).sum()), "malignant": int((y == 1).sum()), "patients": int(d.patient.nunique())}
    print(f"CMMD {s}: {x.shape} benign={summ[s]['benign']} malignant={summ[s]['malignant']} patients={summ[s]['patients']}", flush=True)
summ["build_minutes"] = (time.time() - t0) / 60
json.dump(summ, open(os.path.join(DATA, "cmmd_summary.json"), "w"), indent=1)
print(f"CMMD cache written ({summ['build_minutes']:.1f} min)")
''')
OUT = os.environ["PA_OUT"]; CK = "/kaggle/working/ckpt"; os.makedirs(CK, exist_ok=True)
ENV = {"PA_TMP": "/tmp/cmmd", "PA_DATA": "/kaggle/working/cmmd_data"}
def walk_find(name, root="/kaggle/input"):
    out = []
    for dp, dns, fns in os.walk(root):
        dns[:] = [d for d in dns if d not in ("CMMD", "jpeg", "U24_synthesis", ".git")]
        if name in fns:
            out.append(os.path.join(dp, name))
    return sorted(out)
h = walk_find("cmmd_clf_s0.pt")
if not h:
    raise SystemExit("!! cmmd_clf_s0.pt not found -> Add Input -> the S8 replication notebook output (latest version)")
CLF = h[0]; print("classifier:", CLF)
SESSION, prev = 1, None
for f in walk_find("S10_state.json"):
    s = json.load(open(f))
    if s.get("session", 0) + 1 > SESSION:
        SESSION, prev = s["session"] + 1, os.path.dirname(os.path.dirname(f))
if prev:
    n = 0
    for sub, dst in (("S10", OUT), ("ckpt", CK)):
        for f in glob.glob(f"{prev}/{sub}/*"):
            t = os.path.join(dst, os.path.basename(f))
            if os.path.isfile(f) and not os.path.exists(t):
                shutil.copy2(f, t); n += 1
    print(f"SESSION {SESSION}: carried forward {n} files from {prev}")
else:
    print("SESSION 1")
subprocess.run([sys.executable, "/tmp/prepare_cmmd.py"], check=True, env={**os.environ, **ENV})   # reuses S8's cmmd_data
r = subprocess.run([sys.executable, "-c", "import torch; print(torch.cuda.device_count())"], capture_output=True, text=True)
NGPU = int(r.stdout.split()[-1]) if r.returncode == 0 else 0
print("GPUs visible:", NGPU)
LOGS = f"{OUT}/logs"; os.makedirs(LOGS, exist_ok=True); RC = {}
def run_one(name, gpu, argv, out_dir=None):
    env = os.environ.copy(); env.update(ENV); env["PYTHONUNBUFFERED"] = "1"; env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
    env["CUDA_VISIBLE_DEVICES"] = str(gpu if NGPU >= 2 else 0)
    if out_dir:
        env["PA_OUT"] = out_dir
    print(f">>> {name} started on GPU {env['CUDA_VISIBLE_DEVICES']} | session {used_h():.2f} h", flush=True)
    with open(f"{LOGS}/{name}.log", "a") as fh:
        p = subprocess.Popen([sys.executable, "-u"] + argv, cwd=WD, env=env, text=True, bufsize=1, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
        for line in p.stdout:
            fh.write(line); fh.flush(); print(line if line.startswith("[") else f"[{name}] {line}", end="", flush=True)
        RC[name] = p.wait()
    print(f"<<< {name} return code {RC[name]} | session {used_h():.2f} h" + ("" if RC[name] == 0 else f"  !! see {LOGS}/{name}.log"), flush=True)
    return RC[name]
print("Cell 3 done.")

In [ ]:
# ===================== CELL 4 — SMOKE TEST (first session only, ~15 min) =====================
SM = "/kaggle/working/smoke"; CKS = f"{SM}/ckpt"
if RUN_SMOKE and SESSION == 1 and not glob.glob(f"{CK}/E*_last.pt"):
    shutil.rmtree(SM, ignore_errors=True); os.makedirs(CKS, exist_ok=True)
    steps = [("smoke_train_e1", ["train_ema.py", "--arm", "smk", "--lam", "0.01", "--epochs", "1", "--smoke_batches", "16", "--snap_every", "1000",
                                 "--snap_epochs", "1", "--log_every", "8", "--ckpt_dir", CKS]),
             ("smoke_resume", ["train_ema.py", "--arm", "smk", "--lam", "0.01", "--epochs", "2", "--smoke_batches", "16", "--snap_every", "1000",
                               "--snap_epochs", "2", "--log_every", "8", "--ckpt_dir", CKS]),
             ("smoke_gen_raw", ["s6_generate.py", "--name", "a1", "--ckpts", f"{CKS}/smk_w_ep002.pt", "--n_per_class", "2", "--steps", "5"]),
             ("smoke_gen_ema", ["s6_generate.py", "--name", "b1", "--ckpts", f"{CKS}/smk_ema_ep002.pt", "--n_per_class", "2", "--steps", "5"]),
             ("smoke_pool_a", ["s10_pool.py", "a", "a1", "a1"]), ("smoke_pool_b", ["s10_pool.py", "b", "b1", "b1"]),
             ("smoke_eval", ["s10_eval.py", "--stage", "val", "--sets", "a,b", "--clf", CLF, "--tag", "smk", "--n_boot", "20"])]
    for name, argv in steps:
        if run_one(name, 0, argv, SM) != 0:
            raise SystemExit(f"SMOKE TEST FAILED at {name} -> send Claude this block + S10/logs/{name}.log")
    log2 = open(f"{LOGS}/smoke_resume.log").read()
    if "EMA restored from checkpoint" not in log2 or not os.path.exists(f"{SM}/smk_val_report.txt"):
        raise SystemExit("SMOKE TEST FAILED: EMA was not restored on resume, or no report")
    print(open(f"{SM}/smk_val_report.txt").read())
    shutil.rmtree(SM, ignore_errors=True); RC.clear(); print(f"SMOKE TEST PASSED | session {used_h():.2f} h")
else:
    print("smoke test skipped (session", SESSION, ")")

In [ ]:
# ===================== CELL 5 — JOB LIST (fixed by the pre-registration) + two-GPU scheduler =====================
def ck(n): return f"{CK}/{n}"
TB = str(BUDGET_H - 0.3)
def gen(name, cks, n, seed):
    return ["s6_generate.py", "--name", name, "--ckpts", ",".join(cks), "--n_per_class", str(n), "--dt", "0", "--noise_seed", str(seed)]
RUNS = [("E0_s42", "0.0", 42), ("E1_s42", "0.01", 42), ("E0_s43", "0.0", 43), ("E1_s43", "0.01", 43)]
JOBS = []
for arm, lam, seed in RUNS:
    se = "25,50" if arm == "E0_s42" else "50"
    JOBS.append((f"train_{arm}", ["train_ema.py", "--arm", arm, "--lam", lam, "--seed", str(seed), "--epochs", str(EPOCHS), "--snap_every", "1000",
                                  "--snap_epochs", se, "--budget_h", TB, "--ckpt_dir", CK], [], ck(f"{arm}_ema_ep050.pt"), 8.5, 0.25))
G = lambda *n: [f"{OUT}/gen_{x}.npz" for x in n]
for pre, n, seed in (("v", VAL_N, VAL_SEED), ("t", TEST_N, TEST_SEED)):
    for arm, _, _ in RUNS:
        for kind, f in (("raw", f"{arm}_w_ep050.pt"), ("ema", f"{arm}_ema_ep050.pt")):
            needs = [ck(f)] + ([f"{OUT}/S10_K2_val.json"] if pre == "t" else [])
            JOBS.append((f"gen_{pre}_{arm}_{kind}", gen(f"{pre}_{arm}_{kind}", [ck(f)], n, seed), needs, f"{OUT}/gen_{pre}_{arm}_{kind}.npz", 0.45 if pre == "v" else 0.65, None))
    if pre == "v":
        for kind, f in (("raw25", "E0_s42_w_ep025.pt"), ("ema25", "E0_s42_ema_ep025.pt")):
            JOBS.append((f"gen_v_E0_s42_{kind}", gen(f"v_E0_s42_{kind}", [ck(f)], n, seed), [ck(f)], f"{OUT}/gen_v_E0_s42_{kind}.npz", 0.45, None))
    for e in ("E0", "E1"):
        for kind in ("raw", "ema"):
            JOBS.append((f"pool_{pre}_{e}_{kind}", ["s10_pool.py", f"{pre}_{e}_{kind}", f"{pre}_{e}_s42_{kind}", f"{pre}_{e}_s43_{kind}"],
                         G(f"{pre}_{e}_s42_{kind}", f"{pre}_{e}_s43_{kind}"), f"{OUT}/gen_{pre}_{e}_{kind}.npz", 0.02, None))
CMP = [("K1", "E0_raw,E0_ema"), ("K2", "E0_ema,E1_ema"), ("K3", "E0_raw,E1_raw")]
for k, sets in CMP:
    v = ",".join(f"v_{s}" for s in sets.split(",")); t = ",".join(f"t_{s}" for s in sets.split(","))
    JOBS.append((f"eval_{k}_val", ["s10_eval.py", "--stage", "val", "--sets", v, "--clf", CLF, "--tag", f"S10_{k}"], G(*v.split(",")),
                 f"{OUT}/S10_{k}_val.json", 0.4, None))
    JOBS.append((f"eval_{k}_test", ["s10_eval.py", "--stage", "test", "--sets", t, "--clf", CLF, "--tag", f"S10_{k}", "--alpha", str(TEST_ALPHA)],
                 G(*t.split(",")) + [f"{OUT}/S10_{k}_val.json"], f"{OUT}/S10_{k}_test.json", 0.45, None))
d2 = "v_E0_s42_raw25,v_E0_s42_ema25,v_E0_s42_raw,v_E0_s42_ema"
JOBS.append(("eval_D2_val", ["s10_eval.py", "--stage", "val", "--sets", d2, "--clf", CLF, "--tag", "S10_D2", "--halves", "0"],
             G(*d2.split(",")), f"{OUT}/S10_D2_val.json", 0.4, None))
NAMES = [j[0] for j in JOBS]; assert len(set(NAMES)) == len(NAMES)
print(f"{len(JOBS)} jobs | already done: {sum(os.path.exists(j[3]) for j in JOBS)}")
lock = threading.Lock(); RUNNING, TRIED, FAILED = set(), set(), set()
def pick():
    left = BUDGET_H - used_h()
    for name, argv, needs, done, est, rmin in JOBS:
        if name in RUNNING or name in TRIED or os.path.exists(done) or not all(os.path.exists(f) for f in needs):
            continue
        if (rmin is not None and left >= rmin + 0.3) or (rmin is None and left >= est + 0.25):
            return name, argv, done
    return None
def worker(gpu):
    while True:
        with lock:
            j = pick()
            if j:
                RUNNING.add(j[0]); TRIED.add(j[0])
            busy = bool(RUNNING)
        if j is None:
            if not busy or BUDGET_H - used_h() < 0.35:
                return
            time.sleep(60); continue
        name, argv, done = j
        try:
            rc = run_one(name, gpu, argv)
        except Exception as e:
            print(f"!! {name} could not run: {e!r}", flush=True); rc = -1
        with lock:
            RUNNING.discard(name)
            if rc != 0:
                FAILED.add(name)
            elif not os.path.exists(done):
                print(f"... {name} stopped before finishing (time budget) -> continues next session", flush=True)
        time.sleep(5)
ths = [threading.Thread(target=worker, args=(g,)) for g in range(max(1, min(NGPU, 2)))]
for t in ths: t.start()
for t in ths: t.join()
print(f"scheduler finished | session {used_h():.2f} h | failed {sorted(FAILED)}")

In [ ]:
# ===================== CELL 6 — cleanup + state + report + zip =====================
ex = os.path.exists
for f in glob.glob(f"{CK}/*_w_latest.pt"):
    os.remove(f)
for arm, _, _ in RUNS:
    if ex(ck(f"{arm}_ema_ep050.pt")) and ex(ck(f"{arm}_last.pt")):
        os.remove(ck(f"{arm}_last.pt"))                          # finished -> free 2.5 GB
TL = f"{OUT}/train_logs"; os.makedirs(TL, exist_ok=True)
for f in glob.glob(f"{CK}/*_history.json"):
    shutil.copy(f, TL)
todo = [j[0] for j in JOBS if not ex(j[3])]
json.dump({"session": SESSION, "n_jobs": len(JOBS), "todo": todo, "failed": sorted(FAILED), "return_codes": RC},
          open(f"{OUT}/S10_state.json", "w"), indent=1)
rep = ["S10 REPORT (rules: S10_preregistration.txt)", "=" * 90, f"session {SESSION} | jobs left {len(todo)} | failed {sorted(FAILED)}"]
for arm, _, _ in RUNS:
    f = f"{CK}/{arm}_history.json"
    if ex(f):
        h = json.load(open(f)); rep.append(f"  {arm}: {len(h)}/{EPOCHS} epochs | last mse {h[-1]['mse']:.5f} | {h[-1]['minutes']:.1f} min/epoch")
for k in ("K1", "K2", "K3", "D2"):
    for st in ("val", "test"):
        f = f"{OUT}/S10_{k}_{st}_report.txt"
        if ex(f):
            rep.append(open(f).read())
open(f"{OUT}/S10_report.txt", "w").write("\n".join(rep) + "\n"); print("\n".join(rep))
z = shutil.make_archive("/kaggle/working/S10_results", "zip", OUT)
print("ZIP:", z, f"({os.path.getsize(z) / 1e6:.1f} MB)")
print(subprocess.run(f"du -sh {CK} {OUT}", shell=True, capture_output=True, text=True).stdout)
print("=" * 90)
print("ALL S10 JOBS DONE -> send S10_results.zip (or this log) to Claude" if not todo else
      f"S10 NOT FINISHED ({len(todo)} jobs left) -> NEXT SESSION: same input + this notebook's latest output -> Save & Run All")
if FAILED:
    print("!! FAILED jobs:", sorted(FAILED))